In [ ]:
import hashlib,json,os,subprocess,sys,tarfile
from pathlib import Path
os.chdir('/kaggle/working')
files=list(Path('/kaggle/input').rglob('payload.bin'))
assert len(files)==1
p=files[0]
assert p.stat().st_size==15670364 and hashlib.sha256(p.read_bytes()).hexdigest()=='73cbe07a1b25c5a8d5656f790404d974fbc6a51929baa352bd76172919120603'
root=Path('/kaggle/working/input');root.mkdir(exist_ok=False)
with tarfile.open(p,'r:gz') as tar:
    members=tar.getmembers()
    assert len(members)<2000 and sum(m.size for m in members)<150*1024**2
    assert all(m.isfile() and not Path(m.name).is_absolute() and '..' not in Path(m.name).parts for m in members)
    tar.extractall(root,filter='data')
manifest=root/'INPUT_MANIFEST.json'
assert hashlib.sha256(manifest.read_bytes()).hexdigest()=='8304d40db8b4f329aec3dec9be7efbe77b1610f844b6088e506955a9ad996b13'
for item in json.loads(manifest.read_text()):
    path=root/item['name']
    assert path.stat().st_size==item['bytes'] and hashlib.sha256(path.read_bytes()).hexdigest()==item['sha256']
subprocess.run([sys.executable,str(root/'cloud_worker.py'),'--root',str(root),'--run-id','k230-camera-xy-multiroute-pilot-20261117-a01','--input-sha','8304d40db8b4f329aec3dec9be7efbe77b1610f844b6088e506955a9ad996b13'],check=True)
